In [ ]:
# -*- coding: utf-8 -*-
## 지엘엔
"""
네이버 카페 f-e 검색 전용 크롤러 (GLN 키워드 + 전체 게시판 + 기간 필터 + 중복방지)
https://cafe.naver.com/f-e/cafes/17373998/menus/0?viewType=L&ta=ARTICLE_COMMENT&page=1&q=GLN
"""

import os, re, csv, time
from datetime import datetime, timedelta
from urllib.parse import quote_plus

import pandas as pd
from bs4 import BeautifulSoup as bs
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# =========================
# 기본 설정
# =========================
NAVER_LOGIN_URL = "https://nid.naver.com/nidlogin.login"
NAVER_ID = os.getenv("NAVER_ID", "아이디 입력") ##"아이디 입력" 에 크롤링할 카페에 가입된 아이디 입력
NAVER_PW = os.getenv("NAVER_PW", "비번") ## "비번 입력" 에 크롤링할 카페에 가입된 아이디 입력

CLUB_ID = "16189029"  ##카페 ID 입력
KEYWORD = "지엘엔" ##검색할 키워드 입력
HEADLESS = False
WAIT_SEC = 15
SLEEP_SEC = 1.0
MAX_PAGES = 300

DATE_FROM = datetime(2023, 11, 11)   ##추출 시작할 날짜 (예: 2020년 01월 01일이면 (2020.01.01)
DATE_TO = datetime(2025, 11, 11, 23, 59, 59) ##추출 종료할 날짜 (예: 2020년 01월 01일이면 (2020.01.01.23.59.59)
OUT_CSV = "cafe_l2_GLN_ko.csv" ##최종 export되는 파일 이름 설정


# =========================
# 드라이버 설정
# =========================
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"])
if HEADLESS:
    chrome_options.add_argument("--headless=new")
browser = webdriver.Chrome(options=chrome_options)
browser.maximize_window()
wait = WebDriverWait(browser, WAIT_SEC)


# =========================
# 유틸 함수
# =========================
def switch_to_cafe_content(driver) -> str:
    """iframe(#cafe_main)이 있으면 전환, 없으면 default"""
    driver.switch_to.default_content()
    try:
        driver.execute_script("window.scrollTo(0, 400);")
    except Exception:
        pass

    deadline = time.time() + 15
    while time.time() < deadline:
        frames = driver.find_elements(By.CSS_SELECTOR, "iframe#cafe_main, frame#cafe_main")
        if frames:
            try:
                WebDriverWait(driver, 5).until(
                    EC.frame_to_be_available_and_switch_to_it((By.CSS_SELECTOR, "iframe#cafe_main, frame#cafe_main"))
                )
                return "iframe"
            except Exception:
                pass
        if driver.find_elements(By.CSS_SELECTOR, "div#app, main, section, div#main-area, div.article-board"):
            if driver.find_elements(By.CSS_SELECTOR, "a[href]"):
                return "default"
        time.sleep(0.3)
    return "default"


def safe_text(el):
    try:
        return el.get_text(strip=True)
    except Exception:
        return ""


DATE_FORMATS = [
    "%Y.%m.%d. %H:%M", "%Y.%m.%d. %p %I:%M", "%Y.%m.%d %H:%M",
    "%Y-%m-%d %H:%M", "%Y.%m.%d.", "%Y-%m-%d"
]


def parse_relative_ko(s: str):
    if not s: return None
    now = datetime.now()
    m = re.search(r"(\d+)\s*분\s*전", s)
    if m: return now - timedelta(minutes=int(m.group(1)))
    m = re.search(r"(\d+)\s*시간\s*전", s)
    if m: return now - timedelta(hours=int(m.group(1)))
    m = re.search(r"(\d+)\s*일\s*전", s)
    if m: return now - timedelta(days=int(m.group(1)))
    if "방금" in s: return now
    return None


def parse_date_any(s: str):
    if not s: return None
    rel = parse_relative_ko(s)
    if rel: return rel
    st = s.replace("오전", "AM").replace("오후", "PM")
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(st, fmt)
        except Exception:
            pass
    try:
        return datetime.fromisoformat(st.replace(".", "-").strip())
    except Exception:
        return None


def in_range(dt):
    return bool(dt) and DATE_FROM <= dt <= DATE_TO


def absolutize(href: str) -> str:
    if not href: return ""
    if href.startswith("http"): return href
    if href.startswith("/"): return "https://cafe.naver.com" + href
    return "https://cafe.naver.com/" + href.lstrip("/")


def build_search_url(page: int) -> str:
    return f"https://cafe.naver.com/f-e/cafes/{CLUB_ID}/menus/0?viewType=L&ta=ARTICLE_COMMENT&page={page}&q={quote_plus(KEYWORD)}"


# =========================
# 로그인
# =========================
browser.get(NAVER_LOGIN_URL)
try:
    wait.until(EC.presence_of_element_located((By.NAME, "id")))
    browser.execute_script("document.getElementsByName('id')[0].value=arguments[0];", NAVER_ID)
    browser.execute_script("document.getElementsByName('pw')[0].value=arguments[0];", NAVER_PW)
    browser.find_element(By.XPATH, '//*[@id="log.login"]').click()
    time.sleep(1.5)
except Exception as e:
    print("[경고] 자동 로그인 실패:", e)
    input("로그인 수동 완료 후 Enter: ")


# =========================
# 수집 루프
# =========================
results = []
page = 1
seen_ids = set()
empty_pages = 0

while page <= MAX_PAGES:
    url = build_search_url(page)
    print(f"[검색목록] p{page}: {url}")
    browser.get(url)
    time.sleep(SLEEP_SEC)

    try:
        switch_to_cafe_content(browser)
    except Exception:
        pass

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[href]")))
    except Exception:
        empty_pages += 1
        if empty_pages >= 2:
            print("[완료] 더 이상 목록이 없습니다.")
            break
        page += 1
        continue

    soup = bs(browser.page_source, "html.parser")
    anchors = soup.select("a[href]")
    post_links = []
    for a in anchors:
        href = a.get("href", "")
        if re.search(rf"/f-e/cafes/{CLUB_ID}/articles/\d+", href) or re.search(r"ArticleRead\.nhn.*clubid=", href):
            post_links.append(absolutize(href))
    post_links = sorted(set(post_links))

    if not post_links:
        empty_pages += 1
        if empty_pages >= 3:
            print("[완료] 결과 없음 종료.")
            break
        page += 1
        continue

    empty_pages = 0

    for href in post_links:
        # ✅ 글 ID 기준 중복 방지
        m = re.search(r"articles/(\d+)", href)
        if not m:
            m = re.search(r"articleid=(\d+)", href)
        article_id = m.group(1) if m else href
        if article_id in seen_ids:
            continue
        seen_ids.add(article_id)

        try:
            browser.get(href)
            time.sleep(SLEEP_SEC)
            try:
                switch_to_cafe_content(browser)
            except Exception:
                pass

            dsoup = bs(browser.page_source, "html.parser")

            # 제목
            title = None
            for sel in ["h3.title_text", "div.title_area h3", "h2.tit", "meta[property='og:title']"]:
                node = dsoup.select_one(sel)
                if node:
                    title = node.get("content") if node.name == "meta" else safe_text(node)
                    if title: break
            title = title or "null"

            # 작성자
            author = None
            for sel in ["button.nickname", "span.nickname", "a.nick", "a.m-tcol-c", "span.writer"]:
                node = dsoup.select_one(sel)
                if node and safe_text(node):
                    author = safe_text(node); break
            author = author or "null"

            # 날짜
            date_txt = None
            for sel in ["span.date", "span.regdate", "div.writer-info span"]:
                node = dsoup.select_one(sel)
                if node and safe_text(node):
                    date_txt = safe_text(node); break
            dt = parse_date_any(date_txt)
            if not in_range(dt):
                continue

            # 본문
            content = None
            for sel in [
                "div.se-main-container",
                "div.ArticleContentBox",
                "div#tbody",
                "div.article_viewer",
                "div.ContentRenderer",
            ]:
                node = dsoup.select_one(sel)
                if node and safe_text(node):
                    content = safe_text(node); break
            content = re.sub(r"\s+", " ", content or "null").strip()

            results.append({
                "article_id": article_id,
                "title": title,
                "author": author,
                "date": dt.strftime("%Y-%m-%d %H:%M:%S") if dt else "null",
                "url": href,
                "content": content,
            })
            print(f"  - [{dt.strftime('%Y-%m-%d') if dt else '?'}] {title[:60]}")

        except Exception as e:
            print("[경고] 글 파싱 실패:", e)
            continue

    page += 1


# =========================
# 저장
# =========================
if results:
    df = pd.DataFrame(results, columns=["article_id", "title", "author", "date", "url", "content"])
    df = df.drop_duplicates(subset=["article_id"], keep="first").reset_index(drop=True)
    df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
    print(f"[저장 완료] {OUT_CSV} (총 {len(df)}건)")
else:
    print("[알림] 수집된 데이터가 없습니다.")
